In [340]:

import random

# Tic-Tac-Toe Game
class TicTacToe:

    def __init__(self):
        self.board = [' '] * 9

    def display(self):
        for i in range(0, 9, 3):
            print("|".join(self.board[i:i+3]))
            if i < 6:
                print("-----")

    def available_moves(self):
        return [i for i, x in enumerate(self.board) if x == ' ']

    def make_move(self, position, player):
        if self.board[position] == ' ':
            self.board[position] = player
            return True
        return False

    def check_winner(self):
        wins = [
            (0,1,2), (3,4,5), (6,7,8),
            (0,3,6), (1,4,7), (2,5,8),
            (0,4,8), (2,4,6)
        ]

        for a, b, c in wins:
            if self.board[a] == self.board[b] == self.board[c] != ' ':
                return self.board[a]

        return None


# Q-Learning Agent
class QLearningAgent:

    def __init__(self):
        self.q = {}
        self.lr = 0.1
        self.discount = 0.9
        self.epsilon = 0.2

    def get_state(self, board):
        return tuple(board)

    def choose_action(self, board):
        state = self.get_state(board)
        moves = [i for i, x in enumerate(board) if x == ' ']

        if random.uniform(0, 1) < self.epsilon:
            return random.choice(moves)

        qvals = [self.q.get((state, a), 0) for a in moves]
        max_q = max(qvals)

        best_moves = [
            a for a in moves
            if self.q.get((state, a), 0) == max_q
        ]

        return random.choice(best_moves)

    def update(self, state, action, reward, next_state, done):

        # Convert lists into tuples
        state = tuple(state)
        next_state = tuple(next_state)

        old_q = self.q.get((state, action), 0)

        if done:
            future_q = 0
        else:
            future_q = max([
                self.q.get((next_state, a), 0)
                for a in range(9)
                if next_state[a] == ' '
            ], default=0)

        self.q[(state, action)] = old_q + self.lr * (
            reward + self.discount * future_q - old_q
        )


# Train agents using self-play
agent_x = QLearningAgent()
agent_o = QLearningAgent()

episodes = 20000

for ep in range(episodes):

    game = TicTacToe()
    current = 'X'
    done = False

    history_x = []
    history_o = []

    while not done:

        state = game.board.copy()

        if current == 'X':
            action = agent_x.choose_action(game.board)
        else:
            action = agent_o.choose_action(game.board)

        game.make_move(action, current)

        winner = game.check_winner()
        next_state = game.board.copy()

        if winner:
            if current == 'X':
                reward_x, reward_o = 1, -1
            else:
                reward_x, reward_o = -1, 1

            done = True

        elif not game.available_moves():
            reward_x, reward_o = 0, 0
            done = True

        else:
            reward_x, reward_o = 0, 0

        if current == 'X':
            history_x.append((state, action, next_state, reward_x))
        else:
            history_o.append((state, action, next_state, reward_o))

        if done:

            # Update X agent
            for s, a, ns, r in history_x[::-1]:
                agent_x.update(s, a, r, ns, True)

            # Update O agent
            for s, a, ns, r in history_o[::-1]:
                agent_o.update(s, a, r, ns, True)

        else:
            current = 'O' if current == 'X' else 'X'


# Test trained agent against human
print("Training completed!")
print("You are X. Computer is O.")
print("Enter positions from 1 to 9.")

game = TicTacToe()
current = 'X'

while True:

    game.display()

    if current == 'X':

        try:
            position = int(input("Enter your move (1-9): ")) - 1
        except ValueError:
            print("Please enter a number between 1 and 9.")
            continue

        if position not in game.available_moves():
            print("Invalid move! Try again.")
            continue

    else:
        position = agent_o.choose_action(game.board)
        print("Computer chooses:", position + 1)

    game.make_move(position, current)

    winner = game.check_winner()

    if winner:
        game.display()
        print(winner, "wins!")
        break

    if not game.available_moves():
        game.display()
        print("Match Draw!")
        break

    current = 'O' if current == 'X' else 'X'


# WINDOWS INSTALLATION / EXECUTION COMMANDS
#
# 1. Check Python installation:
#    python --version
#
# 2. No external libraries are required.
#    The program uses Python's built-in random library.
#
# 3. Save the code as:
#    tic_tac_toe.py
#
# 4. Run the program:
#    python tic_tac_toe.py


Training completed!
You are X. Computer is O.
Enter positions from 1 to 9.
 | | 
-----
 | | 
-----
 | | 


Enter your move (1-9):  1


X| | 
-----
 | | 
-----
 | | 
Computer chooses: 7
X| | 
-----
 | | 
-----
O| | 


Enter your move (1-9):  9


X| | 
-----
 | | 
-----
O| |X
Computer chooses: 8
X| | 
-----
 | | 
-----
O|O|X


Enter your move (1-9):  5


X| | 
-----
 |X| 
-----
O|O|X
X wins!
